# H SAF on CF0360: super-obs vs thinning, in observation space

Three ways to turn the same raw H121 obs into assimilated obs, all predicted from the **same** model background:

| configuration | GEOSldas setting | what reaches the EnKF |
|---|---|---|
| **raw mode** | `superob_grid_deg = 0`, `thin_dist_km = 0` | raw obs averaged per CF0360 tile (≈ 1 per tile) |
| **super-obs** | `superob_grid_deg = 0.25`, new footprint (`9624356`) | one obs per 0.25° cell, at the mean position, Gaussian footprint C = 12.5²·I + S |
| **thinning, 20 and 25 km** | `thin_dist_km = 20` / `25` (`a499c55` + `5e220ff`) | raw obs with a tile, greedily thinned to a minimum distance, then averaged per tile |

**Data** (2019-05-01, windows 03z–21z, Metop-A/B/C; fetched from Discover into `~/Desktop/M21C_land_local/`):
raw H121 files; `SFMC_FCST` (ensemble mean) of `M21C_test_CF0360_baseline_HSAF_xc03125` for each window; that run's
ObsFcstAna, used to check the operators and to recover the obs scaling.

**Obs scaling.** The H SAF obs are scaled per tile to the model climatology (z-score), so in model units each obs is
`a_t + b_t · raw`. The scaling files are not local, but ObsFcstAna gives both terms for every tile that received a raw-mode
obs: `b_t = √obsvar / errstd` (the obs-error std is scaled by the same factor) and `a_t = obs − b_t · raw`. These are
applied to every configuration's obs in the tile it is assigned to.

**What this is and is not.** All configurations share one background, so this compares them **in obs space**: how many
obs, how far apart, how they are predicted, and their O − F. It does not show the impact on the analysis or on later
forecasts; that needs cycling runs.

Runs in the `regrid` env in about 3 minutes.

## 0. Settings and setup

In [ ]:
import datetime as dt
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from scipy import sparse
from scipy.spatial import cKDTree

%matplotlib inline
T_START = time.time()

DATA = Path('/Users/amfox/Desktop/M21C_land_local/cf0360_obs_fcst_20190501/cf0360_obs_fcst_20190501')
TILE_RASTER = Path('/Users/amfox/Desktop/M21C_land_local/superob_fov/tile_raster_kansas.npz')     # optional, maps
WINDOWS = [dt.datetime(2019, 5, 1, h) for h in (3, 6, 9, 12, 15, 18, 21)]
THIN_DISTS = [20.0, 25.0]                  # thin_dist_km values to compare
SUPEROB_GRID = 0.25                        # superob_grid_deg
REGION = (36.5, 38.5, -100.5, -98.0)       # SW Kansas, for maps
MAP_TIME = dt.datetime(2019, 5, 1, 3)      # window and satellite for the maps
MAP_SAT = 'metop_a'

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'common' / 'python' / 'io'))
from read_GEOSldas import read_tilecoord
TILECOORD = Path('/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/land_sweeper/LS_OLv8_M36/output/'
                 'SMAP_EASEv2_M36_GLOBAL/rc_out/M21C_test_CF0360.ldas_tilecoord.bin')
FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'superob_vs_thinning'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SATS = ['metop_a', 'metop_b', 'metop_c']
SAT_LABEL = {'metop_a': 'Metop-A', 'metop_b': 'Metop-B', 'metop_c': 'Metop-C'}
DT_ASSIM = dt.timedelta(hours=3)
FOV_RAW_KM = 12.5
MAPL_RADIUS_KM = 6371.0
KM_PER_DEG = np.pi / 180.0 * MAPL_RADIUS_KM
KM2DEG = 1.0 / KM_PER_DEG
FAC_SEARCH = 2.0
N_LAND = 1188020

CONFIGS = ['raw mode', 'super-obs'] + [f'thin {r:g} km' for r in THIN_DISTS]
CCOL = dict(zip(CONFIGS, ['0.35', 'tab:green', 'tab:blue', 'tab:purple', 'tab:orange']))

### Tiles, reader, tile assignment, footprints

Copied from `cf0360_hsaf_superob_fov_v2.ipynb` (single-precision details kept): reader QC and window (with the reader's
64 s TT−UTC offset), obs-to-tile assignment (pert grid, nearest centre of mass, 12.5 km tolerance), the raw-obs footprint
(R: 12.5 km Gaussian, 25 km cut-off) and the new super-ob footprint (N: Gaussian with C = 12.5²·I + S, q ≤ 4).

In [ ]:
tc = read_tilecoord(str(TILECOORD))
COM_LON = tc['com_lon'].astype(np.float32)
COM_LAT = tc['com_lat'].astype(np.float32)
BBOX = {k: tc[k].astype(np.float32) for k in ['min_lon', 'max_lon', 'min_lat', 'max_lat']}
AREA = tc['area'].astype(np.float64)
TREE = cKDTree(np.c_[COM_LON, COM_LAT].astype(np.float64))
assert np.abs(np.load(DATA / 'tile_lat.npy') - tc['com_lat']).max() < 1e-4, 'tile table differs from the run'

TT_MINUS_UTC = 64.184
UNIX0 = dt.datetime(1970, 1, 1)

def window_files(sat, t):
    out = []
    for f in sorted((DATA / 'H121' / sat).glob('*.nc')):
        start = dt.datetime.strptime(f.name.split('_')[-6], '%Y%m%d%H%M%S')
        if t - DT_ASSIM / 2 - dt.timedelta(hours=1) < start <= t + DT_ASSIM / 2:
            out.append(f)
    return out

def read_h121(f, sat, w_low, w_up):
    with nc.Dataset(f) as d:
        d.set_auto_maskandscale(False)
        v = {k: d[k][:] for k in ['latitude', 'longitude', 'time', 'surface_soil_moisture', 'surface_flag',
                                  'processing_flag', 'backscatter40_flag', 'wetland_fraction',
                                  'topographic_complexity', 'subsurface_scattering_probability',
                                  'surface_soil_moisture_sensitivity']}
    s8 = lambda a: a.astype(np.int8).astype(np.int32)
    ssm = v['surface_soil_moisture'].astype(np.int32)
    t = v['time'] * 86400.0
    ok = ((ssm >= 0) & (ssm <= 10000) & (t > w_low) & (t <= w_up)
          & ((v['surface_flag'] & 1) == 0) & ((v['processing_flag'] & 3) == 0)
          & ((v['backscatter40_flag'] & 4) == 0)
          & (s8(v['wetland_fraction']) >= 0) & (s8(v['wetland_fraction']) < 10)
          & (s8(v['topographic_complexity']) >= 0) & (s8(v['topographic_complexity']) < 10)
          & (s8(v['subsurface_scattering_probability']) < 5)
          & (v['surface_soil_moisture_sensitivity'] * 1e-7 > 1.0))
    return pd.DataFrame({'sat': sat, 'lat': v['latitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'lon': v['longitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'sm': (ssm[ok] * 0.01 / 100).astype(np.float32)})

def read_window(t):
    lo = (t - DT_ASSIM / 2 - UNIX0).total_seconds() + TT_MINUS_UTC
    return pd.concat([read_h121(f, s, lo, lo + DT_ASSIM.total_seconds()) for s in SATS for f in window_files(s, t)],
                     ignore_index=True)                      # file order within each satellite

NLON_P, NLAT_P = 1440, 1080
DLON_P, DLAT_P = np.float32(360 / NLON_P), np.float32(180 / NLAT_P)

def pert_cell(lat, lon):
    i = np.ceil((np.asarray(lon, np.float32) - np.float32(-180)) / DLON_P).astype(np.int64)
    j = np.ceil((np.asarray(lat, np.float32) - np.float32(-90)) / DLAT_P).astype(np.int64)
    return np.clip(j, 1, NLAT_P) * NLON_P + np.clip(i, 1, NLON_P)

_key = pert_cell(COM_LAT, COM_LON)
_order = np.argsort(_key, kind='stable')
_sorted_key = _key[_order]

def assign_tiles(lat, lon, fov_km=FOV_RAW_KM):
    """1-based land tile per obs (get_tile_num_for_obs); -1 no tile in pert cell, -2 dropped"""
    lat, lon = np.asarray(lat, np.float32), np.asarray(lon, np.float32)
    k = pert_cell(lat, lon)
    a, b = np.searchsorted(_sorted_key, k), np.searchsorted(_sorted_key, k, side='right')
    t = np.full(len(lat), -1); dx = np.full(len(lat), np.inf, np.float32); dy = dx.copy()
    for n in np.flatnonzero(b > a):
        cand = _order[a[n]:b[n]]
        ddx, ddy = np.abs(lon[n] - COM_LON[cand]), np.abs(lat[n] - COM_LAT[cand])
        best = np.argmin(ddx + ddy)
        t[n], dx[n], dy[n] = cand[best], ddx[best], ddy[best]
    tt = np.clip(t, 0, None)
    outside = ((lon < BBOX['min_lon'][tt]) | (lon > BBOX['max_lon'][tt]) |
               (lat < BBOX['min_lat'][tt]) | (lat > BBOX['max_lat'][tt]))
    mdy = np.float32(fov_km * KM2DEG)
    mdx = (mdy / np.cos(np.radians(lat))).astype(np.float32)
    return np.where(t < 0, -1, np.where(outside & ((dx > mdx) | (dy > mdy)), -2, t + 1))

def _candidates(lat, lon, rx, ry):
    cand = TREE.query_ball_point(np.c_[lon, lat].astype(np.float64), r=np.maximum(rx, ry))
    obs = np.repeat(np.arange(len(lat)), [len(c) for c in cand])
    tile = np.concatenate(cand).astype(np.int64) if len(obs) else np.zeros(0, np.int64)
    dlon = COM_LON[tile].astype(np.float64) - lon[obs]
    dlat = COM_LAT[tile].astype(np.float64) - lat[obs]
    return obs, tile, dlon, dlat, (dlon / rx[obs]) ** 2 + (dlat / ry[obs]) ** 2

def _to_matrix(obs, tile, w, n):
    W = sparse.csr_matrix((w, (obs, tile)), shape=(n, N_LAND))
    s = np.asarray(W.sum(1)).ravel()
    return sparse.diags(np.where(s > 0, 1 / np.where(s > 0, s, 1), 0)) @ W

def fov_R(lat, lon, fov_km=FOV_RAW_KM):
    lat, lon = np.asarray(lat, np.float64), np.asarray(lon, np.float64)
    ry = np.full(len(lat), FAC_SEARCH * fov_km * KM2DEG); rx = ry / np.cos(np.radians(lat))
    obs, tile, _, _, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    return _to_matrix(obs[k], tile[k], np.exp(-0.5 * FAC_SEARCH ** 2 * nd2[k]) * AREA[tile[k]], len(lat))

def fov_N(lat, lon, S, fov_km=FOV_RAW_KM):
    lat, lon, S = np.asarray(lat, np.float64), np.asarray(lon, np.float64), np.asarray(S, np.float64)
    cxx, cyy, cxy = fov_km ** 2 + S[:, 0], fov_km ** 2 + S[:, 1], S[:, 2]
    det = cxx * cyy - cxy ** 2
    lmax = 0.5 * (cxx + cyy) + np.sqrt(0.25 * (cxx - cyy) ** 2 + cxy ** 2)
    ry = FAC_SEARCH * np.sqrt(lmax) * KM2DEG; rx = ry / np.cos(np.radians(lat))
    obs, tile, dlon, dlat, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    obs, tile, dlon, dlat = obs[k], tile[k], dlon[k], dlat[k]
    dx, dy = dlon * KM_PER_DEG * np.cos(np.radians(lat[obs])), dlat * KM_PER_DEG
    q = (cyy[obs] * dx ** 2 - 2 * cxy[obs] * dx * dy + cxx[obs] * dy ** 2) / det[obs]
    k = q <= FAC_SEARCH ** 2
    return _to_matrix(obs[k], tile[k], np.exp(-0.5 * q[k]) * AREA[tile[k]], len(lat))

In [ ]:
def tile_average(df):
    """reader accumulation: one obs per (sat, tile): mean sm, lat, lon of the obs assigned to that tile"""
    k = df[df.tile > 0]
    key = k.sat.map({s: i for i, s in enumerate(SATS)}).to_numpy().astype(np.int64) * (N_LAND + 1) + k.tile.to_numpy()
    uk, inv = np.unique(key, return_inverse=True)
    n = np.bincount(inv)
    m = lambda x: np.bincount(inv, np.asarray(x, np.float64)) / n
    return pd.DataFrame({'sat': np.array(SATS)[uk // (N_LAND + 1)], 'tile': uk % (N_LAND + 1),
                         'lat': m(k.lat), 'lon': m(k.lon), 'sm': m(k.sm), 'n_raw': n})

def superobs(raw, grid=SUPEROB_GRID):
    """super-obs per (sat, cell): mean sm and position, spread S [km^2], tile (12.5 km), reader's collision rule"""
    n_lon, n_lat = int(round(360 / grid)), int(round(180 / grid))
    lon = np.mod(raw.lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0
    lat = raw.lat.to_numpy(np.float64)
    i = np.minimum(n_lon, ((lon + 180.0) / grid).astype(int) + 1)
    j = np.minimum(n_lat, ((lat + 90.0) / grid).astype(int) + 1)
    sat_id = raw.sat.map({s: k for k, s in enumerate(SATS)}).to_numpy()
    uk, inv = np.unique((sat_id.astype(np.int64) * (n_lat + 1) + j) * (n_lon + 1) + i, return_inverse=True)
    N = np.bincount(inv)
    dlon, dlat = lon - (-180.0 + (i - 0.5) * grid), lat - (-90.0 + (j - 0.5) * grid)
    s = lambda x: np.bincount(inv, x) / N
    so = pd.DataFrame({'sat': np.array(SATS)[uk // ((n_lat + 1) * (n_lon + 1))],
                       'lat': s(lat).astype(np.float32), 'lon': s(lon).astype(np.float32),
                       'sm': s(raw.sm.to_numpy(np.float64)), 'n_raw': N})
    mdlon, mdlat = s(dlon), s(dlat)
    coslat = np.cos(np.pi / 180.0 * so.lat.to_numpy(np.float64))
    so['Sxx'] = np.maximum(0.0, s(dlon * dlon) - mdlon ** 2) * (KM_PER_DEG * coslat) ** 2
    so['Syy'] = np.maximum(0.0, s(dlat * dlat) - mdlat ** 2) * KM_PER_DEG ** 2
    so['Sxy'] = (s(dlon * dlat) - mdlon * mdlat) * KM_PER_DEG ** 2 * coslat
    so['tile'] = assign_tiles(so.lat, so.lon)
    t0 = np.clip(so.tile.to_numpy() - 1, 0, None)
    dl = np.abs(so.lon.to_numpy(np.float64) - COM_LON[t0].astype(np.float64)); dl = np.minimum(dl, 360 - dl)
    d2 = dl ** 2 + (so.lat.to_numpy(np.float64) - COM_LAT[t0].astype(np.float64)) ** 2
    holder = {}
    for n, (sat, t, dd) in enumerate(zip(so.sat.to_numpy(), so.tile.to_numpy(), d2)):
        if t > 0:
            h = holder.get((sat, t))
            if h is None or dd < np.float64(np.float32(h[1])):
                holder[sat, t] = (n, dd)
    keep = np.zeros(len(so), bool); keep[[n for n, _ in holder.values()]] = True
    return so[keep].reset_index(drop=True)

def to_xyz(lat, lon):
    la, lo = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    return np.column_stack((np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)))

def thin_min_dist(lat, lon, r_km):
    """greedy: visit obs in order, keep unless closer than r_km to an obs kept earlier (as GEOSldas thin_min_dist)"""
    n = len(lat); keep = np.zeros(n, bool)
    if n == 0:
        return keep
    p = to_xyz(lat, lon); chord = 2 * np.sin(r_km / MAPL_RADIUS_KM / 2)
    pr = cKDTree(p).query_pairs(chord, output_type='ndarray')
    if len(pr):
        pr = pr[((p[pr[:, 0]] - p[pr[:, 1]]) ** 2).sum(1) < chord ** 2]
    E = sparse.csr_matrix((np.ones(len(pr), bool), (pr.max(1), pr.min(1))), shape=(n, n))
    for k in range(n):
        keep[k] = not keep[E.indices[E.indptr[k]:E.indptr[k + 1]]].any()
    return keep

def nn_km(lat, lon):
    if len(lat) < 2:
        return np.full(len(lat), np.nan)
    p = to_xyz(lat, lon); d, _ = cKDTree(p).query(p, 2)
    return 2 * np.arcsin(d[:, 1] / 2) * MAPL_RADIUS_KM

def read_ofa(t):
    f = sorted(DATA.glob(f'*ObsFcstAna.{t:%Y%m%d_%H}00z.nc4'))[0]
    with nc.Dataset(f) as d:
        df = pd.DataFrame({v: np.asarray(d[v][:]) for v in ['species', 'tilenum', 'lat', 'lon', 'obs', 'obsvar',
                                                             'fcst', 'fcstvar']})
        descr = [str(x).strip() for x in (nc.chartostring(d['obsparam_descr'][:]) if d['obsparam_descr'].ndim > 1
                                          else d['obsparam_descr'][:])]
        errstd = np.asarray(d['obsparam_errstd'][:])
    sat_of = {k + 1: f'metop_{s[-4].lower()}' for k, s in enumerate(descr) if s.startswith('ASCAT_HSAF_MET')}
    err = {sat_of[k]: errstd[k - 1] / 100 for k in sat_of}      # [% saturation] -> fraction
    df = df[df.species.isin(sat_of)].copy()
    df['sat'] = df.species.map(sat_of)
    return df, err

## 1. Raw mode, checked against GEOSldas, and the obs scaling

For each window: read the raw obs with the reader's QC, assign them to tiles, average per (satellite, tile), and match
to the run's ObsFcstAna. Then

- **forecast check:** the raw-obs footprint applied to `SFMC_FCST` must reproduce ObsFcstAna `fcst`;
- **scaling:** `b = √obsvar / errstd`, `a = obs − b · raw`, per (satellite, tile, window).

ObsFcstAna holds only obs that pass the model-side screening (e.g. frozen or snow-covered ground), so raw-mode obs
missing from it were screened out. That screening depends on the model state, which varies smoothly, so a tile counts as
**usable** in a window if any H SAF obs in its CF0360 grid cell passed it; the same rule is applied to every
configuration. Obs whose tile has no recovered scaling (no raw-mode obs on that tile in any window) are left out and
counted separately.

In [ ]:
RAW, OFA, FIELD, SCAL, CHECK = {}, {}, {}, [], []
for t in WINDOWS:
    raw = read_window(t)
    raw['tile'] = assign_tiles(raw.lat, raw.lon)
    RAW[t] = raw
    FIELD[t] = np.load(DATA / f'sfmc_fcst_{t:%Y%m%d_%H}00z.npy').astype(np.float64)
    ofa, err = read_ofa(t)
    OFA[t] = ofa
    rm = tile_average(raw)
    m = rm.merge(ofa, left_on=['sat', 'tile'], right_on=['sat', 'tilenum'], suffixes=('', '_ofa'))
    pred = fov_R(m.lat_ofa, m.lon_ofa) @ FIELD[t]
    dpos = np.hypot(m.lat - m.lat_ofa, m.lon - m.lon_ofa)
    CHECK.append({'window': f'{t:%H}z', 'QC raw obs': len(raw), 'raw-mode obs': len(rm), 'in ObsFcstAna': len(ofa),
                  'matched': len(m), 'position diff > 1e-4°': int((dpos > 1e-4).sum()),
                  'max |pred - fcst|': np.abs(pred - m.fcst).max(), 'median |pred - fcst|': np.median(np.abs(pred - m.fcst))})
    m['b'] = np.sqrt(m.obsvar) / m.sat.map(err)
    m['a'] = m.obs - m.b * m.sm
    m['window'] = t
    SCAL.append(m[['window', 'sat', 'tile', 'a', 'b', 'sm', 'obs', 'fcst']])
CHECK = pd.DataFrame(CHECK).set_index('window')
SCAL = pd.concat(SCAL, ignore_index=True)
print(CHECK.to_string(float_format=lambda v: f'{v:.1e}'))

# scaling consistency: (a, b) of one window predicts the scaled obs of the same (sat, tile) in other windows
first = SCAL.sort_values('window').drop_duplicates(['sat', 'tile']).set_index(['sat', 'tile'])
later = SCAL.join(first[['a', 'b', 'window']], on=['sat', 'tile'], rsuffix='0')
later = later[later.window != later.window0]
err_s = np.abs(later.a0 + later.b0 * later.sm - later.obs)
print(f'\nscaling check on {len(later):,} later obs of the same (sat, tile): median |error| {np.median(err_s):.1e}, '
      f'{(err_s > 1e-3).mean():.2%} off by more than 1e-3')

# (a, b) lookup: the same (window, sat, tile) first, else the median over windows for (sat, tile), else over satellites
AB_W = SCAL.set_index(['window', 'sat', 'tile'])[['a', 'b']]
AB = SCAL.groupby(['sat', 'tile'])[['a', 'b']].median()
AB_TILE = SCAL.groupby('tile')[['a', 'b']].median()

# model-side screening (frozen / snow-covered ground) varies smoothly, so a tile counts as usable in a window if any H SAF
# obs in its CF0360 grid cell passed the screening (is in that window's ObsFcstAna)
CELL = (tc['j_indg'].astype(np.int64) - 1) * 360 + tc['i_indg'].astype(np.int64)      # cube cell of each land tile
USABLE_CELLS = {t: set(CELL[OFA[t].tilenum.to_numpy() - 1]) for t in WINDOWS}

**Takeaway:** the raw-obs footprint applied to the run's `SFMC_FCST` reproduces GEOSldas's `fcst`, and the recovered
linear scaling reproduces the scaled obs of the same tile in other windows, so the obs and predictions below are in the
same units as in GEOSldas.

## 2. The configurations

For each window and satellite, from the same raw obs:

- **raw mode:** raw obs with a tile, averaged per tile;
- **super-obs:** averaged per 0.25° cell; tile assigned at the mean position (12.5 km tolerance); one super-ob per tile
  (collision rule); predicted with the new footprint N;
- **thinning:** following `5e220ff`, only raw obs **with a tile** are thinned (file order), then averaged per tile;
  predicted with the raw-obs footprint R at the tile-mean position.

Then each obs is scaled with its tile's (a, b), kept only if its tile is usable in that window, and predicted from that
window's `SFMC_FCST`. O − F is in m³/m³.

In [ ]:
def finish_config(df, t, W):
    """scale, screen and predict; df has sat, tile, lat, lon, sm, n_raw"""
    df = df[df.tile > 0].copy()
    df['window'] = t
    a = np.full(len(df), np.nan); b = a.copy()
    for tab, key in [(AB_W, list(zip([t] * len(df), df.sat, df.tile))), (AB, list(zip(df.sat, df.tile))),
                     (AB_TILE, df.tile.to_numpy())]:
        v = tab.reindex(key)
        fill = ~np.isfinite(a)
        a[fill], b[fill] = v.a.to_numpy()[fill], v.b.to_numpy()[fill]
    df['a'], df['b'] = a, b
    df['screened'] = ~np.isin(CELL[df.tile.to_numpy() - 1], list(USABLE_CELLS[t]))
    df['no_scaling'] = ~np.isfinite(a)
    df['usable'] = ~df.screened & ~df.no_scaling
    df['O'] = df.a + df.b * df.sm
    df['F'] = W(df) @ FIELD[t]
    df['OmF'] = df.O - df.F
    return df

OBS, TIMING = {c: [] for c in CONFIGS}, {}
for t in WINDOWS:
    raw = RAW[t]
    T0 = time.time()
    OBS['raw mode'].append(finish_config(tile_average(raw), t, lambda d: fov_R(d.lat, d.lon)))
    so = superobs(raw)
    OBS['super-obs'].append(finish_config(so, t, lambda d: fov_N(d.lat, d.lon, d[['Sxx', 'Syy', 'Sxy']].to_numpy())))
    for r in THIN_DISTS:
        parts = []
        for sat in SATS:
            k = raw[(raw.sat == sat) & (raw.tile > 0)]                    # file order, obs with a tile only
            parts.append(k[thin_min_dist(k.lat.to_numpy(np.float64), k.lon.to_numpy(np.float64), r)])
        kept = pd.concat(parts, ignore_index=True)
        OBS[f'thin {r:g} km'].append(finish_config(tile_average(kept), t, lambda d: fov_R(d.lat, d.lon)))
    TIMING[t] = time.time() - T0
OBS = {c: pd.concat(v, ignore_index=True) for c, v in OBS.items()}
ALL = {c: d[d.usable].reset_index(drop=True) for c, d in OBS.items()}

# check: raw-mode O and F equal ObsFcstAna obs and fcst where matched
chk = pd.concat([ALL['raw mode'][ALL['raw mode'].window == t].merge(OFA[t], left_on=['sat', 'tile'],
                                                                    right_on=['sat', 'tilenum'])
                 for t in WINDOWS])
print(f'raw mode vs ObsFcstAna ({len(chk):,} obs): max |O - obs| {np.abs(chk.O - chk.obs).max():.1e} '
      f'(median {np.median(np.abs(chk.O - chk.obs)):.1e}), max |F - fcst| {np.abs(chk.F - chk.fcst).max():.1e}')
print(f'built in {sum(TIMING.values()):.0f} s')

In [ ]:
rows = {}
for c, d in ALL.items():
    nn = np.concatenate([nn_km(g.lat.to_numpy(), g.lon.to_numpy()) for _, g in d.groupby(['window', 'sat'])])
    o = OBS[c]
    rows[c] = {'obs (all windows)': len(d), 'per window': len(d) / len(WINDOWS),
               'dropped: screened': f'{o.screened.mean():.1%}',
               'dropped: no scaling': f'{(o.no_scaling & ~o.screened).mean():.1%}',
               'raw obs per obs': d.n_raw.mean(), 'raw obs used': d.n_raw.sum(),
               'NN spacing median [km]': np.nanmedian(nn), 'NN 5th pct [km]': np.nanpercentile(nn, 5),
               'pairs < 15 km [% of obs]': 100 * np.mean(nn < 15)}
COUNTS = pd.DataFrame(rows).T
print('all windows (03z-21z), Metop-A/B/C, same-satellite nearest neighbours')
print(COUNTS.to_string(float_format=lambda v: f'{v:,.1f}'))

**Takeaway:** raw mode keeps essentially every raw obs as its own obs (~1.1 raw obs per obs, ~12 km apart). Super-obs
and thinning end up with a similar number of obs (~22–27k per window, ~21–26 km apart), but super-obs **average** ~3 raw
obs each, while thinning **keeps one raw obs and discards the rest** (~70–75% of them). Thinning removes all pairs closer
than its distance; super-obs leave a few (mean positions across cell edges).

(The higher "no scaling" drop for super-obs is an artefact of this offline analysis: the scaling can only be recovered for
tiles that got a raw-mode obs, and a super-ob's tile often did not. GEOSldas would scale and keep them.)

## 3. What each configuration looks like

The same window and satellite over SW Kansas: every obs coloured by O − F, on the CF0360 tiles. Grey dots are the raw
QC'd obs.

In [ ]:
HAVE_RASTER = TILE_RASTER.exists()
if HAVE_RASTER:
    _r = np.load(TILE_RASTER)
    RAS = (_r['tile'], _r['lat_e'], _r['lon_e'])

def tile_background(ax, box):
    if not HAVE_RASTER:
        return
    tile, lat_e, lon_e = RAS
    r0, r1 = np.searchsorted(lat_e, box[0], 'right') - 1, np.searchsorted(lat_e, box[1])
    c0, c1 = np.searchsorted(lon_e, box[2], 'right') - 1, np.searchsorted(lon_e, box[3])
    tile, lat_e, lon_e = tile[r0:r1, c0:c1], lat_e[r0:r1 + 1], lon_e[c0:c1 + 1]
    land = tile <= N_LAND
    ax.pcolormesh(lon_e, lat_e, np.where(land, 0, 1), cmap=mcolors.ListedColormap(['0.95', '#cfe3f3']),
                  vmin=-0.5, vmax=1.5, rasterized=True, zorder=0)
    ids = np.where(land, tile, -1)
    r, c = np.nonzero(ids[:, 1:] != ids[:, :-1])
    v = np.stack([np.c_[lon_e[c + 1], lat_e[r]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    r, c = np.nonzero(ids[1:, :] != ids[:-1, :])
    h = np.stack([np.c_[lon_e[c], lat_e[r + 1]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    ax.add_collection(LineCollection(np.concatenate([v, h]), colors='0.8', lw=0.4, zorder=0.5))

def inbox(d, box):
    return d[d.lat.between(box[0], box[1]) & d.lon.between(box[2], box[3])]

raw_m = inbox(RAW[MAP_TIME][RAW[MAP_TIME].sat == MAP_SAT], REGION)
fig, axs = plt.subplots(1, len(CONFIGS), figsize=(5.2 * len(CONFIGS), 5.6), constrained_layout=True)
norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-0.12, vmax=0.12)
for ax, c in zip(axs, CONFIGS):
    tile_background(ax, REGION)
    d = inbox(ALL[c][(ALL[c].window == MAP_TIME) & (ALL[c].sat == MAP_SAT)], REGION)
    ax.plot(raw_m.lon, raw_m.lat, '.', color='0.55', ms=2, zorder=1)
    sc = ax.scatter(d.lon, d.lat, c=d.OmF, cmap='RdBu', norm=norm, s=18 + 6 * d.n_raw, edgecolor='k', lw=0.4, zorder=2)
    ax.set_xlim(REGION[2], REGION[3]); ax.set_ylim(REGION[0], REGION[1])
    ax.set_aspect(1 / np.cos(np.radians(np.mean(REGION[:2]))))
    ax.set_title(f'{c}: {len(d)} obs\nO−F mean {d.OmF.mean():+.3f}, std {d.OmF.std():.3f}', fontsize=11)
fig.colorbar(sc, ax=axs, shrink=0.8, label='O − F [m³/m³]')
fig.suptitle(f'{SAT_LABEL[MAP_SAT]}, {MAP_TIME:%Y-%m-%d %H}z, SW Kansas (marker size ∝ raw obs averaged)', fontsize=12)
fig.savefig(FIG_DIR / 'maps_omf.png', dpi=150)
plt.show()

## 4. O − F statistics

Pooled over all windows and satellites, on usable tiles. In raw mode and with thinning almost every obs is one raw obs;
super-obs average several, which lowers the random part of the obs error (and of the representativeness error), so
std(O − F) should drop as the number of raw obs per super-ob grows. The last row of the first table is restricted to the
**tiles shared by all configurations**, so the configurations are compared on the same places.

In [ ]:
def omf_stats(d):
    x = d.OmF
    return pd.Series({'obs': len(d), 'O−F mean': x.mean(), 'O−F std': x.std(), 'O−F rms': np.sqrt((x ** 2).mean()),
                      '|O−F| p95': x.abs().quantile(0.95)})

STATS = pd.DataFrame({c: omf_stats(d) for c, d in ALL.items()}).T
common = set.intersection(*[set(zip(d.window, d.sat, d.tile)) for d in ALL.values()])
STATS_C = pd.DataFrame({c: omf_stats(d[[k in common for k in zip(d.window, d.sat, d.tile)]]) for c, d in ALL.items()}).T
print('all usable obs'); print(STATS.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'\nonly the {len(common):,} (window, sat, tile) present in every configuration')
print(STATS_C.to_string(float_format=lambda v: f'{v:.4f}'))

grp = lambda n: np.where(n >= 6, '6+', n.astype(str))
so = ALL['super-obs'].assign(n=lambda d: grp(d.n_raw))
BYN = so.groupby('n').OmF.agg(['size', 'mean', 'std'])
print('\nsuper-obs: O−F by number of raw obs averaged'); print(BYN.to_string(float_format=lambda v: f'{v:.4f}'))

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
bins = np.linspace(-0.2, 0.2, 161)
for c, d in ALL.items():
    axs[0].hist(d.OmF, bins=bins, histtype='step', lw=2, color=CCOL[c], density=True,
                label=f'{c}: std {d.OmF.std():.3f}')
axs[0].set_xlabel('O − F [m³/m³]'); axs[0].set_ylabel('density'); axs[0].legend(fontsize=9)
axs[1].errorbar(np.arange(len(BYN)), BYN['std'], fmt='o-', color=CCOL['super-obs'], label='super-obs')
axs[1].axhline(ALL['raw mode'].OmF.std(), color=CCOL['raw mode'], ls='--', label='raw mode (all obs)')
axs[1].set_xticks(np.arange(len(BYN))); axs[1].set_xticklabels(BYN.index)
axs[1].set_xlabel('raw obs averaged into the super-ob'); axs[1].set_ylabel('std(O − F) [m³/m³]'); axs[1].legend()
axs[1].set_title('averaging reduces the random part of O − F')
fig.savefig(FIG_DIR / 'omf_stats.png', dpi=150)
plt.show()

## 5. Spatial correlation of O − F

For pairs of obs of the **same satellite in the same window**, the correlation of their O − F (after removing each
window/satellite mean) against their distance. Correlated O − F between neighbours comes from correlated obs errors
(overlapping raw footprints, shared retrieval errors) and from background errors that are correlated in space. The EnKF
treats H SAF obs errors as correlated only through `xcorr = 0.3125°`, so close pairs with strongly correlated O − F are
over-counted information.

The curves show *how correlated* neighbours are at each distance (Pearson correlation of the two O − F values over all
pairs in a 5 km bin; bins with fewer than 200 pairs are left out); the bars show *how many* close pairs each
configuration has.

In [ ]:
EDGES = np.arange(0, 102.5, 5.0)

MIN_PAIRS = 200

def pair_corr(d, max_km=100.0):
    # Pearson correlation of O-F between the two obs of each pair, per distance bin (pairs counted in both orders)
    nb = len(EDGES) - 1
    S1, S2, S11, S12, n = (np.zeros(nb) for _ in range(5))
    for _, g in d.groupby(['window', 'sat']):
        x = g.OmF.to_numpy() - g.OmF.mean()
        p = to_xyz(g.lat, g.lon)
        pr = cKDTree(p).query_pairs(2 * np.sin(max_km / MAPL_RADIUS_KM / 2), output_type='ndarray')
        if not len(pr):
            continue
        dk = 2 * np.arcsin(np.sqrt(((p[pr[:, 0]] - p[pr[:, 1]]) ** 2).sum(1)) / 2) * MAPL_RADIUS_KM
        bi = np.digitize(dk, EDGES) - 1
        ok = (bi >= 0) & (bi < nb)
        bi, u, v = bi[ok], x[pr[ok, 0]], x[pr[ok, 1]]
        S1 += np.bincount(bi, u + v, nb); S11 += np.bincount(bi, u * u + v * v, nb)
        S12 += np.bincount(bi, 2 * u * v, nb); n += 2 * np.bincount(bi, minlength=nb)
    m = S1 / np.maximum(n, 1)
    var = S11 / np.maximum(n, 1) - m ** 2
    rho = (S12 / np.maximum(n, 1) - m ** 2) / np.where(var > 0, var, np.nan)
    return np.where(n / 2 >= MIN_PAIRS, rho, np.nan), n / 2

CORR = {c: pair_corr(d) for c, d in ALL.items()}
mid = 0.5 * (EDGES[:-1] + EDGES[1:])
fig, axs = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for c, (rho, cnt) in CORR.items():
    axs[0].plot(mid, rho, 'o-', color=CCOL[c], label=c, ms=4)
    axs[1].bar(mid + (CONFIGS.index(c) - 1.5) * 1.1, cnt / len(WINDOWS), width=1.1, color=CCOL[c], label=c)
axs[0].axhline(0, color='k', lw=0.6)
axs[0].set_xlabel('distance between two obs (same satellite, same window) [km]'); axs[0].set_ylabel('correlation of O − F')
axs[0].legend(); axs[0].set_title('how correlated neighbouring innovations are')
axs[1].set_xlim(0, 50); axs[1].set_xlabel('distance [km]'); axs[1].set_ylabel('pairs per window')
axs[1].legend(); axs[1].set_title('how many close pairs each configuration has')
fig.savefig(FIG_DIR / 'omf_pair_correlation.png', dpi=150)
plt.show()

near = pd.DataFrame({c: {'pairs < 20 km per window': cnt[mid < 20].sum() / len(WINDOWS),
                         'O−F corr, 10-20 km': np.nanmean(rho[(mid > 10) & (mid < 20)]) if np.isfinite(rho[(mid > 10) & (mid < 20)]).any() else np.nan,
                         'O−F corr, 25-30 km': rho[(mid > 25) & (mid < 30)][0],
                         'O−F corr, 50-55 km': rho[(mid > 50) & (mid < 55)][0]}
                     for c, (rho, cnt) in CORR.items()}).T
print(near.to_string(float_format=lambda v: f'{v:,.2f}'))

**Takeaway:** O − F is strongly correlated in space for every configuration: ~0.67 at 25–30 km, still ~0.5 at 50 km and
~0.35 at 100 km. At the same distance, thinned obs are exactly as correlated as raw obs (the curves coincide); thinning
only removes the closest pairs. Super-obs are *more* correlated at a given distance, because averaging removes part of
the uncorrelated noise and leaves the spatially coherent part (background error and correlated obs error). So neither
20–25 km thinning nor 0.25° super-obs gives independent innovations; most of the correlation is at scales beyond
either. Whether that matters depends on how much of it is background error (handled by the ensemble, HPHᵀ) versus obs
error (which R treats as correlated only over `xcorr` = 0.3125°): that split cannot be made from O − F alone.

**Takeaway (section 4):** averaging ~3 raw obs lowers std(O − F) only a little: 0.031 for super-obs vs 0.033 for raw mode
and thinning (on the same tiles: 0.0312 vs 0.0321–0.0326), and it barely changes from 2 to 6+ raw obs per super-ob. So
the random, uncorrelated part of the raw-obs error is small compared with the rest of O − F: neighbouring raw obs (12.5 km
apart, ~25 km footprints) have largely shared errors, and averaging them does not cancel much. That also means a super-ob
is not much more accurate than one raw obs, so keeping its error std equal to `errstd` (as GEOSldas does) is reasonable.
Thinned obs have the same O − F statistics as raw obs, as expected.

## 6. Summary

In [ ]:
s = COUNTS.join(STATS[['O−F mean', 'O−F std']]).join(near)
print(s[['obs (all windows)', 'raw obs per obs', 'raw obs used', 'NN spacing median [km]', 'pairs < 20 km per window',
         'O−F mean', 'O−F std', 'O−F corr, 25-30 km']].to_string(float_format=lambda v: f'{v:,.3f}'))
print(f'\nruntime {time.time() - T_START:.0f} s')

**Caveats**

- Obs space only, against one background: no analysis, no cycling, so no forecast impact. The decisive test is the
  cycling runs on Discover (CF0360 control vs super-obs vs thinning).
- In GEOSldas the obs-error std of a super-ob is the same as that of one raw obs (`errstd`, scaled), although its random
  error is smaller (section 4); thinned obs carry the raw-obs error correctly but carry less information in total.
- The obs scaling is recovered from ObsFcstAna (exact where checked); obs on tiles that received no raw-mode obs in a
  window are left out, in every configuration ("tile not usable").
- One day (7 windows), early May: many high-latitude obs are screened out (frozen/snow), so the statistics are dominated
  by mid-latitudes.